# Analisis Ulasan: Apa yang Dikeluhkan Pembeli?

Notebook ini memakai data ulasan bersih dari `05_data_prep_ulasan.ipynb` untuk riset pasar dari sisi pembeli. Pertanyaan yang dijawab:

1. Kategori mana yang paling banyak mendapat ulasan negatif?
2. Tema apa yang paling sering dikeluhkan pembeli, misalnya pengiriman, kualitas, atau kesesuaian pesanan?
3. Apakah tema keluhan berbeda antarkategori?
4. Produk mana yang paling banyak dikeluhkan, dan mana yang paling konsisten mendapat ulasan positif?

Hal yang perlu diingat:

- Ulasan bintang 3 sudah dibuang di tahap data prep, jadi yang dibandingkan adalah ulasan negatif (bintang 1 dan 2) dengan ulasan positif (bintang 4 dan 5).
- Data berasal dari tahun 2019 dan hanya dari 158 toko, sehingga hasilnya menggambarkan toko-toko tersebut, bukan seluruh Tokopedia.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

df = pd.read_csv("../Dataset/processed/ulasan_bersih.csv")
df["teks_bersih"] = df["teks_bersih"].fillna("")
print(df.shape)
df.head()

In [ ]:
# Gaya grafik, sama dengan notebook sebelumnya
BIRU = "#2a78d6"
ORANYE = "#eb6834"
ABU = "#c8c6bf"
TEKS = "#52514e"

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#8a8983",
    "axes.labelcolor": TEKS,
    "xtick.color": TEKS,
    "ytick.color": TEKS,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8,
})

## 1. Ulasan negatif per kategori

In [ ]:
per_kategori = (
    df.groupby("kategori")
    .agg(
        jumlah_ulasan=("label", "size"),
        jumlah_negatif=("label", lambda s: (s == "negatif").sum()),
    )
)
per_kategori["persen_negatif"] = per_kategori["jumlah_negatif"] / per_kategori["jumlah_ulasan"] * 100
per_kategori = per_kategori.sort_values("persen_negatif")
per_kategori.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
warna = [ORANYE if k == "handphone" else BIRU for k in per_kategori.index]
ax.barh(per_kategori.index, per_kategori["persen_negatif"], color=warna, height=0.6)
for y, v in enumerate(per_kategori["persen_negatif"]):
    ax.text(v + 0.1, y, f"{v:.1f}%", va="center", fontsize=9, color=TEKS)
ax.set_xlim(0, per_kategori["persen_negatif"].max() * 1.2)
ax.set_title("Persentase ulasan negatif per kategori")
ax.set_xlabel("Ulasan negatif (%)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

Handphone adalah kategori dengan ulasan negatif tertinggi (7,6%), sekitar enam kali lipat kategori elektronik (1,2%). Tetapi angka ini perlu dicek lebih jauh: apakah keluhannya merata di semua produk handphone, atau terpusat di beberapa produk saja?

In [ ]:
negatif_hp = df[(df["kategori"] == "handphone") & (df["label"] == "negatif")]
produk_teratas = negatif_hp["nama_produk"].value_counts().head(5)
print(f"Ulasan negatif handphone: {len(negatif_hp)}")
print(f"Porsi dari 2 produk teratas: {produk_teratas.head(2).sum() / len(negatif_hp):.0%}")
produk_teratas

In [ ]:
hp = df[df["kategori"] == "handphone"]
tanpa_dua_teratas = hp[~hp["nama_produk"].isin(produk_teratas.index[:2])]
print(f"Persen negatif handphone tanpa 2 produk teratas: {(tanpa_dua_teratas['label'] == 'negatif').mean():.1%}")

Dua produk, yaitu sebuah headset bluetooth dan ponsel Nokia 130, menyumbang 38% dari seluruh ulasan negatif di kategori handphone. Kalau kedua produk ini dikeluarkan, persentase negatif handphone turun dari 7,6% menjadi 6,0%. Angka itu masih tertinggi dibanding kategori lain, jadi kategori handphone memang lebih sering dikeluhkan. Tetapi sebagian besar selisihnya berasal dari dua produk tersebut.

Kategori "handphone" di dataset ini juga berisi produk yang bukan ponsel, seperti headset, tablet tulis LCD, dan sabuk penyangga punggung. Kategori ini sebaiknya dibaca sebagai "handphone dan aksesoris gadget".

## 2. Tema yang dibahas dalam ulasan

Setiap ulasan diberi tanda tema berdasarkan kata kunci. Satu ulasan bisa punya lebih dari satu tema. Contohnya, "barang bagus tapi pengiriman lama" masuk tema kualitas produk dan pengiriman.

Cara ini sederhana dan mudah dijelaskan, tetapi kasar. Kata "sesuai" misalnya muncul di "barang sesuai pesanan" (pujian) maupun "barang tidak sesuai" (keluhan). Karena itu, yang dilihat bukan sekadar seberapa sering tema disebut, tetapi perbandingan seberapa sering tema itu muncul di ulasan negatif dibanding di ulasan positif.

In [ ]:
TEMA = {
    "Pengiriman": ["kirim", "dikirim", "pengiriman", "kurir", "ekspedisi", "jne", "jnt", "sicepat",
                   "ongkir", "datang", "sampai", "nyampe", "lama", "telat", "terlambat", "cepat"],
    "Kemasan": ["packing", "bubble", "wrap", "kardus", "dus", "bungkus", "kemasan", "dikemas",
                "packaging", "penyok"],
    "Kualitas produk": ["kualitas", "awet", "kuat", "rusak", "jelek", "mati", "pecah", "cacat",
                        "berfungsi", "fungsi", "ori", "original", "palsu", "kw", "bahan", "tipis",
                        "tebal", "patah", "bocor", "lecet"],
    "Kesesuaian pesanan": ["sesuai", "deskripsi", "gambar", "foto", "beda", "berbeda", "warna",
                           "ukuran", "size", "model", "kurang", "salah", "tertukar"],
    "Pelayanan penjual": ["respon", "respons", "seller", "penjual", "admin", "chat", "ramah", "balas",
                          "dibalas", "pelayanan", "komunikatif", "responsif"],
    "Harga": ["harga", "murah", "mahal", "worth", "diskon", "promo", "hemat"],
}
daftar_tema = list(TEMA)

kata_ulasan = df["teks_bersih"].str.split().map(set)
for tema, kata_kunci in TEMA.items():
    df[tema] = kata_ulasan.map(lambda kata: not kata.isdisjoint(kata_kunci))

print(f"Ulasan tanpa tema apa pun: {(~df[daftar_tema].any(axis=1)).mean():.0%}")

In [ ]:
negatif = df[df["label"] == "negatif"]
positif = df[df["label"] == "positif"]

tema_label = pd.DataFrame({
    "persen_di_negatif": negatif[daftar_tema].mean() * 100,
    "persen_di_positif": positif[daftar_tema].mean() * 100,
})
tema_label["rasio"] = tema_label["persen_di_negatif"] / tema_label["persen_di_positif"]
tema_label = tema_label.sort_values("rasio")
tema_label.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
posisi = np.arange(len(tema_label))
tinggi = 0.38
ax.barh(posisi + tinggi / 2, tema_label["persen_di_negatif"], height=tinggi, color=ORANYE, label="Ulasan negatif")
ax.barh(posisi - tinggi / 2, tema_label["persen_di_positif"], height=tinggi, color=BIRU, label="Ulasan positif")
ax.set_yticks(posisi, labels=tema_label.index)
ax.set_title("Seberapa sering setiap tema disebut")
ax.set_xlabel("Persen ulasan yang menyebut tema")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

Cara membaca grafik: kalau batang oranye lebih panjang dari batang biru, tema itu lebih sering muncul di ulasan negatif, artinya tema tersebut cenderung menjadi sumber keluhan.

Temuan:

- Kualitas produk adalah sumber keluhan terbesar. Tema ini disebut di 28% ulasan negatif, dibanding hanya 16% ulasan positif (1,8 kali lebih sering).
- Kesesuaian pesanan disebut di sekitar sepertiga ulasan, baik negatif maupun positif. Tema ini sering muncul sebagai keluhan ("tidak sesuai gambar"), tetapi juga sebagai pujian ("barang sesuai deskripsi").
- Pengiriman adalah tema yang paling sering disebut secara keseluruhan, tetapi justru sedikit lebih sering di ulasan positif. Pembeli lebih sering memuji pengiriman yang cepat daripada mengeluhkan pengiriman yang lambat.
- Pelayanan penjual dan kemasan jauh lebih sering disebut di ulasan positif. Kedua tema ini lebih sering menjadi alasan pujian daripada keluhan.

In [ ]:
# Contoh ulasan negatif untuk setiap tema
with pd.option_context("display.max_colwidth", 110):
    for tema in ["Kualitas produk", "Kesesuaian pesanan", "Pengiriman"]:
        print(tema)
        display(negatif.loc[negatif[tema], ["kategori", "teks"]].sample(4, random_state=3))

## 3. Tema keluhan per kategori

Untuk setiap kategori, dihitung persentase ulasan negatif yang menyebut setiap tema. Angka ini menunjukkan apa yang paling sering dikeluhkan di masing-masing kategori.

In [ ]:
keluhan_kategori = negatif.groupby("kategori")[daftar_tema].mean() * 100
keluhan_kategori.round(0)

In [ ]:
peta_warna = LinearSegmentedColormap.from_list("oranye", ["#fdf1ea", ORANYE])

fig, ax = plt.subplots(figsize=(8.5, 3.6))
gambar = ax.imshow(keluhan_kategori.values, cmap=peta_warna, vmin=0, vmax=50, aspect="auto")
ax.set_xticks(range(len(daftar_tema)), labels=daftar_tema, rotation=20, ha="right")
ax.set_yticks(range(len(keluhan_kategori)), labels=keluhan_kategori.index)
for i in range(keluhan_kategori.shape[0]):
    for j in range(keluhan_kategori.shape[1]):
        ax.text(j, i, f"{keluhan_kategori.iloc[i, j]:.0f}%", ha="center", va="center", fontsize=9, color="#0b0b0b")
ax.grid(False)
ax.set_title("Tema yang disebut dalam ulasan negatif, per kategori")
fig.colorbar(gambar, ax=ax, shrink=0.8, label="Persen ulasan negatif")
plt.tight_layout()
plt.show()

In [ ]:
negatif["kategori"].value_counts()

Temuan:

- Fashion: keluhan paling banyak tentang kesesuaian pesanan (47%), misalnya ukuran yang tidak pas atau warna yang berbeda dari foto. Masuk akal, karena pakaian dan sepatu sulit dinilai hanya dari foto.
- Elektronik dan handphone: kualitas produk jauh lebih sering dikeluhkan (sekitar sepertiga keluhan), misalnya barang mati, tidak berfungsi, atau suaranya jelek.
- Olahraga: kombinasi kesesuaian pesanan (41%) dan kualitas (28%).
- Pengiriman disebut di 30 sampai 42% ulasan negatif di semua kategori. Angka ini tinggi, tetapi seperti terlihat di bagian 2, pengiriman juga sama seringnya disebut di ulasan positif. Banyak ulasan negatif menyebut pengiriman hanya sebagai keterangan, misalnya "barang sudah sampai tapi rusak". Jadi angka ini tidak berarti pengiriman adalah keluhan utama.
- Pertukangan hanya punya 24 ulasan negatif, sehingga persentasenya kurang bisa diandalkan.

Implikasi untuk penjual: di kategori fashion, tabel ukuran dan foto yang akurat kemungkinan paling membantu mengurangi keluhan. Di kategori elektronik dan handphone, pengecekan kualitas sebelum barang dikirim lebih penting.

## 4. Produk yang paling banyak dikeluhkan dan paling konsisten dipuji

Catatan penting: bagian ini bukan sistem rekomendasi. Sistem rekomendasi yang sesungguhnya butuh data siapa membeli apa, supaya bisa menyarankan produk berdasarkan kemiripan selera antarpembeli. Dataset ini tidak punya informasi pembeli, jadi yang bisa dibuat hanya peringkat produk berdasarkan ulasannya.

Hanya produk dengan minimal 20 ulasan yang dihitung. Produk dengan 2 ulasan yang keduanya positif akan tampak "100% positif", padahal buktinya terlalu sedikit.

In [ ]:
MIN_ULASAN = 20

per_produk = (
    df.groupby("id_produk")
    .agg(
        nama_produk=("nama_produk", "first"),
        kategori=("kategori", "first"),
        jumlah_ulasan=("label", "size"),
        persen_negatif=("label", lambda s: (s == "negatif").mean() * 100),
    )
)
per_produk = per_produk[per_produk["jumlah_ulasan"] >= MIN_ULASAN]

print(f"Produk dengan minimal {MIN_ULASAN} ulasan: {len(per_produk)}")
print(f"Produk tanpa satu pun ulasan negatif: {(per_produk['persen_negatif'] == 0).mean():.0%}")

In [ ]:
# Produk yang paling banyak dikeluhkan
with pd.option_context("display.max_colwidth", 70):
    display(per_produk.sort_values("persen_negatif", ascending=False).head(10).round(1))

In [ ]:
# Produk paling konsisten dipuji di setiap kategori:
# tanpa ulasan negatif, diurutkan dari yang ulasannya paling banyak
konsisten = (
    per_produk[per_produk["persen_negatif"] == 0]
    .sort_values("jumlah_ulasan", ascending=False)
    .groupby("kategori")
    .head(3)
    .sort_values(["kategori", "jumlah_ulasan"], ascending=[True, False])
)
with pd.option_context("display.max_colwidth", 70):
    display(konsisten[["kategori", "nama_produk", "jumlah_ulasan"]])

Temuan:

- Separuh lebih produk dengan minimal 20 ulasan tidak punya satu pun ulasan negatif. Ini sejalan dengan temuan sebelumnya bahwa sebagian besar ulasan sangat positif.
- Produk yang paling banyak dikeluhkan didominasi kategori handphone dan elektronik, termasuk dua produk yang disebut di bagian 1. Keduanya punya ratusan ulasan, jadi persentase negatifnya bisa dipercaya.
- Produk yang "paling konsisten dipuji" diurutkan berdasarkan jumlah ulasan, karena produk dengan ratusan ulasan tanpa keluhan memberi bukti yang lebih kuat daripada produk dengan 20 ulasan tanpa keluhan.

## Kesimpulan

1. Handphone adalah kategori dengan ulasan negatif tertinggi (7,6%). Namun 38% keluhannya berasal dari hanya dua produk.
2. Kualitas produk adalah sumber keluhan terbesar secara keseluruhan. Pengiriman, pelayanan penjual, dan kemasan justru lebih sering dipuji daripada dikeluhkan.
3. Keluhan berbeda antarkategori. Di fashion, keluhan didominasi ketidaksesuaian ukuran dan warna. Di elektronik dan handphone, keluhan didominasi kualitas barang.
4. Separuh lebih produk yang ulasannya cukup banyak tidak pernah mendapat ulasan negatif. Keluhan terpusat pada segelintir produk.

Keterbatasan:

- Tema ditentukan dari kata kunci, jadi kasar. Satu kata bisa bermakna pujian atau keluhan tergantung konteksnya, dan 28% ulasan tidak masuk tema mana pun.
- Ulasan bintang 3, yang sering berisi keluhan ringan, tidak ikut dianalisis.
- Peringkat produk bukan sistem rekomendasi, karena dataset tidak punya data pembeli.
- Data berasal dari 158 toko pada tahun 2019.